# 垃圾分类 — ResNet18 + ResNet34 迁移学习

本项目采用 **PyTorch** 框架，基于预训练的 **ResNet18** 和 **ResNet34** 进行迁移学习，
替代原有的 MindSpore MobileNetV2 方案。两个模型分别通过两阶段训练（冻结骨干网络 → 全网络微调），
最终以集成推理（ensemble）方式结合两个模型的预测结果，并辅以水平翻转 TTA（Test-Time Augmentation）
来提升分类精度，实现 26 类垃圾的准确识别。

## 1. 环境与导入

In [ ]:
import os
import json
import time
import random
import numpy as np
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import datasets, transforms, models

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. 训练配置

In [ ]:
# ===== 数据路径 =====
DATASET_ROOT = "./datasets/5fbdf571c06d3433df85ac65-momodel/garbage_26x100"
TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR   = os.path.join(DATASET_ROOT, "val")
SAVE_DIR  = "./results"

# ===== 训练超参数 =====
IMG_DIM       = 224
TRAIN_BATCH   = 32
LOADER_WORKERS = 4

PHASE1_EPOCHS = 5      # 阶段1: 冻结骨干，只训练 fc
PHASE2_EPOCHS = 20     # 阶段2: 解冻全网络微调

LR_PHASE1     = 1e-3
LR_PHASE2     = 3e-5
WEIGHT_DECAY  = 1e-4

SEED = 42

os.makedirs(SAVE_DIR, exist_ok=True)
print("Config loaded.")

## 3. 数据加载

In [ ]:
def fix_random_state(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def prepare_dataloader():
    """构建包含 train+val 的完整训练集（数据增强）。"""
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(IMG_DIM, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.05
        ),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.12), ratio=(0.3, 3.3)),
    ])

    train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
    val_ds   = datasets.ImageFolder(VAL_DIR,   transform=train_transform)

    merged_ds = ConcatDataset([train_ds, val_ds])

    loader = DataLoader(
        merged_ds,
        batch_size=TRAIN_BATCH,
        shuffle=True,
        num_workers=LOADER_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    return train_ds, merged_ds, loader


# 构建数据
fix_random_state(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
base_ds, merged_ds, loader = prepare_dataloader()
num_classes = len(base_ds.classes)
print(f'Classes: {num_classes}, Total samples: {len(merged_ds)}')

## 4. 模型定义与训练

In [ ]:
def init_resnet18(num_classes):
    """构建 ResNet18，替换最后一层全连接。"""
    net = models.resnet18(pretrained=True)
    net.fc = nn.Linear(net.fc.in_features, num_classes)
    return net


def init_resnet34(num_classes):
    """构建 ResNet34，替换最后一层全连接。"""
    net = models.resnet34(pretrained=True)
    net.fc = nn.Linear(net.fc.in_features, num_classes)
    return net


def lock_backbone(net):
    """冻结骨干网络，只保留 fc 层可训练。"""
    for p in net.parameters():
        p.requires_grad = False
    for p in net.fc.parameters():
        p.requires_grad = True


def unlock_all_layers(net):
    """解冻全部参数。"""
    for p in net.parameters():
        p.requires_grad = True


def run_single_epoch(net, loader, criterion, optimizer, device):
    """训练一个 epoch，返回 (loss, acc)。"""
    net.train()
    accum_loss = 0.0
    accum_correct = 0
    total = 0

    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        out = net(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()

        _, preds = torch.max(out, 1)
        bs = labels.size(0)
        accum_loss += loss.item() * bs
        accum_correct += torch.sum(preds == labels).item()
        total += bs

    return accum_loss / total, accum_correct / total


def run_two_phase_training(net, loader, device, net_name='ResNet'):
    """两阶段训练：Phase1 冻结骨干 → Phase2 全网络微调。"""
    net = net.to(device)

    try:
        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    except TypeError:
        criterion = nn.CrossEntropyLoss()

    # Phase 1
    print(f'\n===== {net_name} Phase 1: train fc only =====')
    lock_backbone(net)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, net.parameters()),
                           lr=LR_PHASE1, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS)

    for ep in range(PHASE1_EPOCHS):
        loss, acc = run_single_epoch(net, loader, criterion, optimizer, device)
        scheduler.step()
        print(f'  Phase1 [{ep+1}/{PHASE1_EPOCHS}] loss={loss:.4f} acc={acc:.4f}')

    # Phase 2
    print(f'\n===== {net_name} Phase 2: fine-tune all layers =====')
    unlock_all_layers(net)
    optimizer = optim.Adam(net.parameters(), lr=LR_PHASE2, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS)

    for ep in range(PHASE2_EPOCHS):
        loss, acc = run_single_epoch(net, loader, criterion, optimizer, device)
        scheduler.step()
        print(f'  Phase2 [{ep+1}/{PHASE2_EPOCHS}] loss={loss:.4f} acc={acc:.4f}')

    return net

## 5. 执行训练

训练 ResNet18（两阶段：冻结骨干 5 epoch → 全网络微调 20 epoch）。
模型已通过 `train_r18.py` 离线训练完成，如需重新训练可取消注释执行。

In [ ]:
# ResNet18 训练 (已通过 train_r18.py 离线完成)
# 如需重新训练，取消注释以下代码执行

# fix_random_state(SEED)
# net18 = init_resnet18(num_classes)
# net18 = run_two_phase_training(net18, loader, device, net_name='ResNet18')
#
# save_path = os.path.join(SAVE_DIR, 'r18_gc26.pth')
# torch.save(net18.state_dict(), save_path)
# print(f'Saved ResNet18 to: {save_path}')

print('ResNet18 训练已离线完成，权重文件: results/r18_gc26.pth')

## 6. 执行 ResNet34 训练

训练 ResNet34（两阶段：冻结骨干 5 epoch → 全网络微调 20 epoch）。
模型已通过 `train_r34.py` 离线训练完成，如需重新训练可取消注释执行。

In [ ]:
# ResNet34 训练 (已通过 train_r34.py 离线完成)
# 如需重新训练，取消注释以下代码执行

# fix_random_state(SEED)
# net34 = init_resnet34(num_classes)
# net34 = run_two_phase_training(net34, loader, device, net_name='ResNet34')
#
# save_path = os.path.join(SAVE_DIR, 'r34_gc26.pth')
# torch.save(net34.state_dict(), save_path)
# print(f'Saved ResNet34 to: {save_path}')

print('ResNet34 训练已离线完成，权重文件: results/r34_gc26.pth')

## 7. 模型推理验证

In [ ]:
# 加载已训练模型，在验证集上简单测试
from torchvision import transforms as T

idx_to_label = {
    0: 'Plastic Bottle', 1: 'Hats', 2: 'Newspaper', 3: 'Cans',
    4: 'Glassware', 5: 'Glass Bottle', 6: 'Cardboard', 7: 'Basketball',
    8: 'Paper', 9: 'Metalware', 10: 'Disposable Chopsticks', 11: 'Lighter',
    12: 'Broom', 13: 'Old Mirror', 14: 'Toothbrush', 15: 'Dirty Cloth',
    16: 'Seashell', 17: 'Ceramic Bowl', 18: 'Paint bucket', 19: 'Battery',
    20: 'Fluorescent lamp', 21: 'Tablet capsules', 22: 'Orange Peel',
    23: 'Vegetable Leaf', 24: 'Eggshell', 25: 'Banana Peel'
}

test_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# 加载 ResNet18
m18 = models.resnet18(pretrained=False)
m18.fc = nn.Linear(m18.fc.in_features, 26)
m18.load_state_dict(torch.load("./results/r18_gc26.pth", map_location=device))
m18.to(device).eval()

# 加载 ResNet34
m34 = models.resnet34(pretrained=False)
m34.fc = nn.Linear(m34.fc.in_features, 26)
m34.load_state_dict(torch.load("./results/r34_gc26.pth", map_location=device))
m34.to(device).eval()

# 测试几张验证集图片
val_root = "./datasets/5fbdf571c06d3433df85ac65-momodel/garbage_26x100/val/00_00/"
if os.path.isdir(val_root):
    files = os.listdir(val_root)[:5]
    for fname in files:
        img = cv2.imread(os.path.join(val_root, fname))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor = test_transform(img_rgb).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = (m18(tensor) + m34(tensor)) / 2.0
            pred = int(torch.argmax(logits, dim=1).item())
        print(f"{fname} -> {idx_to_label[pred]}")
else:
    print("验证集目录不存在，请检查路径。")

## 8. 作业评分

===========================================
**模型预测代码答题区域**
===========================================

In [ ]:
## 生成 main.py 时请勾选此 cell

import numpy as np
import cv2
import torch
import torch.nn as nn
from torchvision import transforms, models

idx_to_label = {0: 'Plastic Bottle', 1: 'Hats', 2: 'Newspaper', 3: 'Cans', 4: 'Glassware', 5: 'Glass Bottle', 6: 'Cardboard', 7: 'Basketball',
            8: 'Paper', 9: 'Metalware', 10: 'Disposable Chopsticks', 11: 'Lighter', 12: 'Broom', 13: 'Old Mirror', 14: 'Toothbrush',
            15: 'Dirty Cloth', 16: 'Seashell', 17: 'Ceramic Bowl', 18: 'Paint bucket', 19: 'Battery', 20: 'Fluorescent lamp', 21: 'Tablet capsules',
            22: 'Orange Peel', 23: 'Vegetable Leaf', 24: 'Eggshell', 25: 'Banana Peel'}

In [ ]:
## 生成 main.py 时请勾选此 cell

PATH_NET18 = "./results/r18_gc26.pth"
PATH_NET34 = "./results/r34_gc26.pth"
IMG_DIM = 224

_hw = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_net18 = None
_net34 = None

_img_pipeline = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_DIM, IMG_DIM)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])


def _make_net18():
    net = models.resnet18(pretrained=False)
    net.fc = nn.Linear(net.fc.in_features, 26)
    return net


def _make_net34():
    net = models.resnet34(pretrained=False)
    net.fc = nn.Linear(net.fc.in_features, 26)
    return net


def _ensure_models():
    global _net18, _net34

    if _net18 is not None and _net34 is not None:
        return

    n18 = _make_net18()
    w18 = torch.load(PATH_NET18, map_location=_hw)
    n18.load_state_dict(w18)
    n18.to(_hw)
    n18.eval()

    n34 = _make_net34()
    w34 = torch.load(PATH_NET34, map_location=_hw)
    n34.load_state_dict(w34)
    n34.to(_hw)
    n34.eval()

    _net18 = n18
    _net34 = n34


def _preprocess(image):
    if not isinstance(image, np.ndarray):
        image = np.array(image)

    if image.ndim != 3:
        raise ValueError("Input image must have shape (H, W, C)")

    if image.shape[2] == 1:
        image = np.repeat(image, 3, axis=2)
    elif image.shape[2] != 3:
        raise ValueError("Input image channel must be 1 or 3")

    if image.dtype != np.uint8:
        image = np.clip(image, 0, 255).astype(np.uint8)

    image = cv2.resize(image, (IMG_DIM, IMG_DIM))
    return image


def predict(image):
    """预测入口：输入 np.ndarray (H, W, C)，返回类别名称字符串。"""
    _ensure_models()
    image = _preprocess(image)

    img_flipped = np.ascontiguousarray(image[:, ::-1, :])

    t_orig = _img_pipeline(image).unsqueeze(0).to(_hw)
    t_flip = _img_pipeline(img_flipped).unsqueeze(0).to(_hw)

    with torch.no_grad():
        logits_18 = (_net18(t_orig) + _net18(t_flip)) / 2.0
        logits_34 = (_net34(t_orig) + _net34(t_flip)) / 2.0
        combined = (logits_18 + logits_34) / 2.0
        pred_idx = int(torch.argmax(combined, dim=1).item())

    return idx_to_label[pred_idx]

## 9. 本地测试

In [ ]:
image_path = './datasets/5fbdf571c06d3433df85ac65-momodel/garbage_26x100/val/00_00/'
import os
files = os.listdir(image_path)
if files:
    image = cv2.imread(os.path.join(image_path, files[0]))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = predict(image)
    print(f"Predicted: {result}")